# Cleaning the Company Employee Dataset

This notebook loads a messy company employee dataset, identifies and quantifies data quality issues (missing values, inconsistent categorical entries, duplicate records), applies appropriate cleaning techniques, verifies the results, and exports the final cleaned dataset.

## 1. Loading and Inspecting the Dataset

In [1]:
import pandas as pd

df = pd.read_csv("Day11_Messy_Company_Employee_Dataset.csv")
print("Dataset loaded successfully.")
df.head()

Dataset loaded successfully.


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,NaN,Data Scientist,48.0,Other,108371.0,13.2,2017-06-13,Pune,5.0,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31.0,Female,108824.0,12.1,2021-09-15,Mumbai,2.0,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28.0,Female,119400.0,13.2,2021-02-05,Bengaluru,3.0,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30.0,Female,111847.0,NaN,2018-08-05,Jaipur,4.0,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25.0,Female,78677.0,10.9,2018-07-24,Hyderabad,4.0,Hybrid


In [2]:
print("Shape (rows, columns):", df.shape)
df.info()

Shape (rows, columns): (157, 12)
<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        157 non-null    str    
 1   Employee_Name      157 non-null    str    
 2   Department         152 non-null    str    
 3   Job_Title          157 non-null    str    
 4   Age                153 non-null    float64
 5   Gender             152 non-null    str    
 6   Annual_Salary      152 non-null    float64
 7   Experience_Years   154 non-null    float64
 8   Joining_Date       157 non-null    str    
 9   City               152 non-null    str    
 10  Performance_Score  154 non-null    float64
 11  Work_Mode          155 non-null    str    
dtypes: float64(4), str(8)
memory usage: 14.8 KB


In [3]:
df.dtypes

Employee_ID              str
Employee_Name            str
Department               str
Job_Title                str
Age                  float64
Gender                   str
Annual_Salary        float64
Experience_Years     float64
Joining_Date             str
City                     str
Performance_Score    float64
Work_Mode                str
dtype: object

In [4]:
# Keep a copy of the original (before cleaning) for later comparison
df_original = df.copy()

## 2. Identifying and Quantifying Data Issues

### 2.1 Missing Values

In [5]:
missing_counts = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing_Count": missing_counts,
    "Missing_Percent": missing_percent
})
missing_summary[missing_summary["Missing_Count"] > 0]

,Missing_Count,Missing_Percent
Department,5,3.18
Age,4,2.55
Gender,5,3.18
Annual_Salary,5,3.18
Experience_Years,3,1.91
City,5,3.18
Performance_Score,3,1.91
Work_Mode,2,1.27


### 2.2 Inconsistent Categorical Entries

Checking for case differences, extra whitespace, or otherwise inconsistent labels within text columns.

In [6]:
categorical_cols = ["Department", "Job_Title", "Gender", "City", "Work_Mode"]

for col in categorical_cols:
    print(f"--- {col} ---")
    print(sorted(df[col].dropna().unique().tolist()))
    print()

--- Department ---
['Customer Success', 'Data & Analytics', 'ENGINEERING', 'Engineering', 'Finance', 'Human Resources', 'Marketing', 'Operations', 'Sales', 'engineering']

--- Job_Title ---
['Accountant', 'BI Analyst', 'Business Development Executive', 'Content Strategist', 'Customer Success Executive', 'Customer Success Manager', 'Data Scientist', 'Finance Manager', 'Financial Analyst', 'HR Executive', 'HR Manager', 'Marketing Executive', 'Marketing Manager', 'Operations Executive', 'Operations Manager', 'Process Analyst', 'QA Engineer', 'Recruiter', 'Sales Executive', 'Sales Manager', 'Senior Software Engineer', 'Software Engineer', 'Support Specialist']

--- Gender ---
['Female', 'MALE', 'Male', 'Other', 'female']

--- City ---
['Bengaluru', 'Chandigarh', 'Chennai', 'DELHI', 'Delhi', 'Delhi ', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai', 'Pune', 'Srinagar', 'delhi']

--- Work_Mode ---
['Hybrid', 'Office', 'REMOTE', 'Remote', 'remote']



We can see inconsistencies caused by mixed case and stray whitespace, for example:
- `Department`: `'Engineering'`, `'ENGINEERING'`, `'engineering'` are the same department written differently.
- `Gender`: `'Male'`, `'MALE'`, `'female'` need to be standardized.
- `City`: `'Delhi'`, `'DELHI'`, `'delhi'`, `'Delhi '` (trailing space) all refer to the same city.
- `Work_Mode`: `'Remote'`, `'REMOTE'`, `'remote'` need standardizing.

### 2.3 Duplicate Records

In [7]:
full_duplicates = df.duplicated().sum()
id_duplicates = df.duplicated(subset=["Employee_ID"]).sum()

print("Fully duplicated rows:", full_duplicates)
print("Rows with duplicate Employee_ID:", id_duplicates)

df[df.duplicated(subset=["Employee_ID"], keep=False)].sort_values("Employee_ID")

Fully duplicated rows: 7
Rows with duplicate Employee_ID: 7


,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
70,EMP0021,Riya Khan,Operations,Operations Executive,49.0,Female,71549.0,4.6,2024-09-20,Chandigarh,3.0,Hybrid
104,EMP0021,Riya Khan,Operations,Operations Executive,49.0,Female,71549.0,4.6,2024-09-20,Chandigarh,3.0,Hybrid
78,EMP0048,Nikhil Dar,Finance,Financial Analyst,27.0,Female,NaN,10.1,2020-06-18,Chandigarh,2.0,Office
110,EMP0048,Nikhil Dar,Finance,Financial Analyst,27.0,Female,NaN,10.1,2020-06-18,Chandigarh,2.0,Office
11,EMP0075,Manya Joshi,Sales,Sales Manager,29.0,Male,92623.0,11.9,2025-08-17,Jaipur,4.0,Hybrid
61,EMP0075,Manya Joshi,Sales,Sales Manager,29.0,Male,92623.0,11.9,2025-08-17,Jaipur,4.0,Hybrid
25,EMP0097,Zoya Shah,Finance,Financial Analyst,41.0,Male,81531.0,4.1,2023-06-05,Mumbai,4.0,Office
55,EMP0097,Zoya Shah,Finance,Financial Analyst,41.0,Male,81531.0,4.1,2023-06-05,Mumbai,4.0,Office
92,EMP0120,Saira Sheikh,Marketing,Marketing Executive,52.0,Male,107328.0,17.4,2020-11-25,Bengaluru,2.0,Remote
156,EMP0120,Saira Sheikh,Marketing,Marketing Executive,52.0,Male,107328.0,17.4,2020-11-25,Bengaluru,2.0,Remote


### 2.4 Data Types

Checking whether numeric columns are stored with the correct data type.

In [8]:
df.dtypes

Employee_ID              str
Employee_Name            str
Department               str
Job_Title                str
Age                  float64
Gender                   str
Annual_Salary        float64
Experience_Years     float64
Joining_Date             str
City                     str
Performance_Score    float64
Work_Mode                str
dtype: object

`Age` and `Performance_Score` are stored as `float64`, but both are logically whole numbers (age in years, a rating score). We'll convert them to integers after handling their missing values, since `int` columns in Pandas can't hold `NaN` directly.

## 3. Cleaning the Dataset

### 3.1 Removing Duplicate Records

Since all duplicate rows are exact full-row duplicates (same Employee_ID and identical data), we drop the repeated copies with `drop_duplicates()`, keeping the first occurrence.

In [9]:
rows_before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
rows_after = len(df)

print(f"Rows before removing duplicates: {rows_before}")
print(f"Rows after removing duplicates: {rows_after}")
print(f"Duplicate rows removed: {rows_before - rows_after}")

Rows before removing duplicates: 157
Rows after removing duplicates: 150
Duplicate rows removed: 7


### 3.2 Standardizing Inconsistent Categorical Entries

Trimming whitespace and standardizing text case so that the same category isn't counted as multiple different values.

In [10]:
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

# Title-case for Department and City; standard capitalization for Gender and Work_Mode
df["Department"] = df["Department"].replace("nan", pd.NA).str.title()
df["City"] = df["City"].replace("nan", pd.NA).str.title()
df["Gender"] = df["Gender"].replace("nan", pd.NA).str.capitalize()
df["Work_Mode"] = df["Work_Mode"].replace("nan", pd.NA).str.capitalize()

for col in categorical_cols:
    print(f"--- {col} (after standardizing) ---")
    print(sorted(df[col].dropna().unique().tolist()))
    print()

--- Department (after standardizing) ---
['Customer Success', 'Data & Analytics', 'Engineering', 'Finance', 'Human Resources', 'Marketing', 'Operations', 'Sales']

--- Job_Title (after standardizing) ---
['Accountant', 'BI Analyst', 'Business Development Executive', 'Content Strategist', 'Customer Success Executive', 'Customer Success Manager', 'Data Scientist', 'Finance Manager', 'Financial Analyst', 'HR Executive', 'HR Manager', 'Marketing Executive', 'Marketing Manager', 'Operations Executive', 'Operations Manager', 'Process Analyst', 'QA Engineer', 'Recruiter', 'Sales Executive', 'Sales Manager', 'Senior Software Engineer', 'Software Engineer', 'Support Specialist']

--- Gender (after standardizing) ---
['Female', 'Male', 'Other']

--- City (after standardizing) ---
['Bengaluru', 'Chandigarh', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Kolkata', 'Mumbai', 'Pune', 'Srinagar']

--- Work_Mode (after standardizing) ---
['Hybrid', 'Office', 'Remote']



### 3.3 Handling Missing Values

Different columns need different strategies, chosen based on what best preserves the dataset's accuracy:

- **`Department`, `Gender`, `City`, `Work_Mode`** (categorical): filled with the **mode** (most frequent value), a standard approach for missing categorical data.
- **`Age`, `Annual_Salary`, `Performance_Score`** (numeric, roughly symmetric): filled with the **mean**.
- **`Experience_Years`** (numeric, can be skewed by a few very senior employees): filled with the **median**, which is more robust to outliers.

In [11]:
# Categorical columns -> fill with mode
for col in ["Department", "Gender", "City", "Work_Mode"]:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)
    print(f"Filled missing '{col}' with mode: '{mode_value}'")

Filled missing 'Department' with mode: 'Engineering'
Filled missing 'Gender' with mode: 'Male'
Filled missing 'City' with mode: 'Pune'


Filled missing 'Work_Mode' with mode: 'Remote'


In [12]:
# Age, Annual_Salary, Performance_Score -> fill with mean
for col in ["Age", "Annual_Salary", "Performance_Score"]:
    mean_value = df[col].mean()
    df[col] = df[col].fillna(round(mean_value, 2))
    print(f"Filled missing '{col}' with mean: {round(mean_value, 2)}")

Filled missing 'Age' with mean: 38.09
Filled missing 'Annual_Salary' with mean: 88842.58
Filled missing 'Performance_Score' with mean: 3.55


In [13]:
# Experience_Years -> fill with median (more robust to outliers/skew)
median_exp = df["Experience_Years"].median()
df["Experience_Years"] = df["Experience_Years"].fillna(median_exp)
print(f"Filled missing 'Experience_Years' with median: {median_exp}")

Filled missing 'Experience_Years' with median: 9.2


In [14]:
# Confirm no missing values remain
df.isnull().sum()

Employee_ID          0
Employee_Name        0
Department           0
Job_Title            0
Age                  0
Gender               0
Annual_Salary        0
Experience_Years     0
Joining_Date         0
City                 0
Performance_Score    0
Work_Mode            0
dtype: int64

### 3.4 Correcting Data Types

Now that `Age` and `Performance_Score` have no missing values, they can be safely converted to integers.

In [15]:
df["Age"] = df["Age"].astype(int)
df["Performance_Score"] = df["Performance_Score"].astype(int)

# Joining_Date should be a proper datetime type, not plain text
df["Joining_Date"] = pd.to_datetime(df["Joining_Date"])

df.dtypes

Employee_ID                     str
Employee_Name                   str
Department                      str
Job_Title                       str
Age                           int64
Gender                          str
Annual_Salary               float64
Experience_Years            float64
Joining_Date         datetime64[us]
City                            str
Performance_Score             int64
Work_Mode                       str
dtype: object

### 3.5 A Note on `dropna()` and Forward Fill

For this dataset, imputing with mean/median/mode was the more sensible choice, since every column had only a small number of missing values (a few percent at most) and dropping those rows would have discarded otherwise valid employee records. `dropna()` would only make sense if a row had too many missing fields to be usable, and forward fill (`ffill()`) suits sequential/time-ordered data more than this kind of independent employee record — but both are demonstrated below for completeness.

In [16]:
# Demonstrating dropna(): would remove any row with at least one missing value
demo_dropna = df_original.dropna()
print(f"If we had used dropna() instead: {len(demo_dropna)} rows would remain out of {len(df_original)} — too aggressive for this dataset.")

# Demonstrating forward fill on a copy, for illustration only
demo_ffill = df_original.copy()
demo_ffill["Work_Mode"] = demo_ffill["Work_Mode"].ffill()
print("Forward-fill example applied to 'Work_Mode' column (not used in the final cleaned dataset).")

If we had used dropna() instead: 125 rows would remain out of 157 — too aggressive for this dataset.
Forward-fill example applied to 'Work_Mode' column (not used in the final cleaned dataset).


## 4. Verifying the Cleaned Dataset

In [17]:
print("Shape after cleaning:", df.shape)
df.info()

Shape after cleaning: (150, 12)
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Employee_ID        150 non-null    str           
 1   Employee_Name      150 non-null    str           
 2   Department         150 non-null    str           
 3   Job_Title          150 non-null    str           
 4   Age                150 non-null    int64         
 5   Gender             150 non-null    str           
 6   Annual_Salary      150 non-null    float64       
 7   Experience_Years   150 non-null    float64       
 8   Joining_Date       150 non-null    datetime64[us]
 9   City               150 non-null    str           
 10  Performance_Score  150 non-null    int64         
 11  Work_Mode          150 non-null    str           
dtypes: datetime64[us](1), float64(2), int64(2), str(7)
memory usage: 14.2 KB


In [18]:
df.isnull().sum()

Employee_ID          0
Employee_Name        0
Department           0
Job_Title            0
Age                  0
Gender               0
Annual_Salary        0
Experience_Years     0
Joining_Date         0
City                 0
Performance_Score    0
Work_Mode            0
dtype: int64

In [19]:
print("Remaining duplicate rows:", df.duplicated().sum())

Remaining duplicate rows: 0


In [20]:
df.describe()

,Age,Annual_Salary,Experience_Years,Joining_Date,Performance_Score
count,150.000000,150.000000,150.000000,150,150.000000
mean,38.086667,88842.582133,8.916667,2021-04-06 15:12:00,3.540000
min,21.000000,43331.000000,0.600000,2017-01-04 00:00:00,2.000000
25%,31.000000,71123.500000,4.450000,2018-12-23 00:00:00,3.000000
50%,39.000000,88842.580000,9.200000,2021-01-05 00:00:00,4.000000
75%,46.000000,104693.250000,13.175000,2023-08-15 18:00:00,4.000000
max,52.000000,162942.000000,17.400000,2026-01-06 00:00:00,5.000000
std,9.244579,22354.495428,4.929658,NaN,0.924172


In [21]:
df.head(10)

,Employee_ID,Employee_Name,Department,Job_Title,Age,Gender,Annual_Salary,Experience_Years,Joining_Date,City,Performance_Score,Work_Mode
0,EMP0098,Ananya Nair,Engineering,Data Scientist,48,Other,108371.0,13.2,2017-06-13,Pune,5,Office
1,EMP0046,Faizan Khan,Marketing,Marketing Manager,31,Female,108824.0,12.1,2021-09-15,Mumbai,2,Office
2,EMP0017,Saira Malik,Human Resources,HR Manager,28,Female,119400.0,13.2,2021-02-05,Bengaluru,3,Remote
3,EMP0105,Neha Menon,Engineering,Software Engineer,30,Female,111847.0,9.2,2018-08-05,Jaipur,4,Hybrid
4,EMP0143,Hiba Menon,Sales,Sales Manager,25,Female,78677.0,10.9,2018-07-24,Hyderabad,4,Hybrid
5,EMP0117,Zoya Kapoor,Finance,Finance Manager,42,Male,128567.0,15.4,2017-08-02,Chennai,4,Hybrid
6,EMP0081,Maryam Mir,Marketing,Content Strategist,37,Male,58878.0,1.8,2024-01-18,Pune,2,Office
7,EMP0026,Omar Das,Engineering,QA Engineer,38,Male,112276.0,12.0,2019-05-14,Chandigarh,4,Hybrid
8,EMP0129,Dev Ahmed,Data & Analytics,Data Scientist,45,Male,135730.0,14.1,2021-01-27,Srinagar,2,Remote
9,EMP0121,Karan Patel,Data & Analytics,BI Analyst,40,Female,101886.0,8.5,2020-06-28,Chennai,4,Remote


## 5. Before vs. After Comparison

In [22]:
comparison = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Missing Values",
        "Duplicate Rows",
        "Age dtype",
        "Performance_Score dtype",
        "Joining_Date dtype",
        "Unique Department labels",
        "Unique Gender labels",
        "Unique City labels",
        "Unique Work_Mode labels"
    ],
    "Before Cleaning": [
        len(df_original),
        df_original.isnull().sum().sum(),
        df_original.duplicated().sum(),
        df_original["Age"].dtype,
        df_original["Performance_Score"].dtype,
        df_original["Joining_Date"].dtype,
        df_original["Department"].nunique(),
        df_original["Gender"].nunique(),
        df_original["City"].nunique(),
        df_original["Work_Mode"].nunique()
    ],
    "After Cleaning": [
        len(df),
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        df["Age"].dtype,
        df["Performance_Score"].dtype,
        df["Joining_Date"].dtype,
        df["Department"].nunique(),
        df["Gender"].nunique(),
        df["City"].nunique(),
        df["Work_Mode"].nunique()
    ]
})
comparison

,Metric,Before Cleaning,After Cleaning
0,Total Rows,157,150
1,Total Missing Values,32,0
2,Duplicate Rows,7,0
3,Age dtype,float64,int64
4,Performance_Score dtype,float64,int64
5,Joining_Date dtype,str,datetime64[us]
6,Unique Department labels,10,8
7,Unique Gender labels,5,3
8,Unique City labels,13,10
9,Unique Work_Mode labels,5,3


## 6. Exporting the Cleaned Dataset

In [23]:
df.to_csv("Cleaned_Company_Employee_Dataset.csv", index=False)
print("Cleaned dataset exported as 'Cleaned_Company_Employee_Dataset.csv'")

Cleaned dataset exported as 'Cleaned_Company_Employee_Dataset.csv'


## 7. Summary of Cleaning Steps Performed

1. **Removed duplicate records:** 7 fully duplicated rows (same `Employee_ID` and identical data across all columns) were removed using `drop_duplicates()`, reducing the dataset from 157 to 150 rows.

2. **Standardized inconsistent categorical entries:** Trimmed stray whitespace and normalized text casing in `Department`, `Gender`, `City`, and `Work_Mode`, so that values like `'ENGINEERING'`, `'engineering'`, and `'Engineering'` (or `'Delhi '` with a trailing space) were consolidated into a single consistent label each.

3. **Handled missing values using targeted strategies:**
   - Categorical columns (`Department`, `Gender`, `City`, `Work_Mode`) — filled with the **mode**.
   - Roughly symmetric numeric columns (`Age`, `Annual_Salary`, `Performance_Score`) — filled with the **mean**.
   - `Experience_Years`, which can be skewed by a few long-tenured employees — filled with the **median** for robustness against outliers.
   - `dropna()` was considered but not used, since it would have discarded valid records over just one or two missing fields; forward-fill was demonstrated but not applicable here since the data isn't sequential.

4. **Corrected data types:** Converted `Age` and `Performance_Score` from `float64` to `int64` (now that missing values were resolved), and converted `Joining_Date` from plain text to a proper `datetime64` type.

5. **Verified the cleaned dataset:** Confirmed zero remaining missing values, zero duplicate rows, correct data types, and a consistent, reduced set of category labels — then compared these metrics directly against the original dataset in a before/after table.

6. **Exported the result:** Saved the final cleaned dataset as `Cleaned_Company_Employee_Dataset.csv`, ready for further analysis.